In [ ]:
# 1. Install specific requirements
!pip install -q fastapi uvicorn pyngrok nest_asyncio pydantic pillow
!pip install -q qwen-vl-utils

In [ ]:
import nest_asyncio
import uvicorn
import base64
import torch
from io import BytesIO
from PIL import Image
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info


In [ ]:
# ==========================================
# 2. Configuration
# ==========================================
NGROK_TOKEN = "" # <--- PASTE YOUR NGROK TOKEN HERE
model_id = "mohamedashraff22/arabic-menu-ocr-v2"

In [ ]:
# 3. Load Model (Exactly as your working code)
print("Loading Model...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, 
    device_map="auto",
    trust_remote_code=True
).eval()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
print("Model Loaded!")

# Exact prompt from your working code
prompt = """Extract every food or drink item and its price from this menu image.
Keep item names and prices exactly as written in the image.
If an item has multiple sizes, list each size as a separate entry.
Only include items that have a visible price. Skip anything without a price.
Return a JSON object with one key "items" containing an array. Each item has "name" (string) and "price" (string)."""

In [ ]:
# 4. FastAPI Setup
app = FastAPI(title="Kaggle Qwen OCR")

class InferenceRequest(BaseModel):
    image_base64: str

@app.post("/generate")
async def generate_text(req: InferenceRequest):
    # Convert incoming base64 back to PIL Image
    image_data = base64.b64decode(req.image_base64)
    raw_image = Image.open(BytesIO(image_data)).convert("RGB")
    
    # Exact message formatting from your working code
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": raw_image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Process & Format exactly as your working code
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate Output exactly as your working code
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=5000, 
            do_sample=False,
            repetition_penalty=1.15  
        )
        
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    return {"status": "success", "data": output}



In [ ]:
# 5. Start Ngrok & Background Server
import threading
import time

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill() 
tunnel = ngrok.connect(8000)

print("\n" + "="*60)
print(f"🚀 COPY THIS URL TO YOUR LOCAL SCRIPT:")
print(f"   {tunnel.public_url}/generate")
print("="*60 + "\n")

# Run Uvicorn in a background thread to avoid asyncio crashes
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Keep the cell alive so the server doesn't shut down
while True:
    time.sleep(100)


🚀 COPY THIS URL TO YOUR LOCAL SCRIPT:
   https://brycen-obliterative-camie.ngrok-free.dev/generate



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# Local Vs code "code"

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
import uvicorn
import base64
import requests
import json_repair

app = FastAPI(title="Local OCR Gateway")

# ==========================================
# PASTE YOUR NGROK URL FROM KAGGLE HERE:
# ==========================================
KAGGLE_URL = "https://brycen-obliterative-camie.ngrok-free.dev/generate"


@app.post("/extract")
async def extract_menu(file: UploadFile = File(...)):
    try:
        # 1. Take image from Postman and encode it
        image_bytes = await file.read()
        base64_image = base64.b64encode(image_bytes).decode("utf-8")

        # 2. Send Base64 image to Kaggle
        payload = {"image_base64": base64_image}
        response = requests.post(KAGGLE_URL, json=payload)

        if response.status_code != 200:
            raise Exception(f"Kaggle Error: {response.text}")

        # 3. Clean and parse the returned JSON string
        raw_output = response.json().get("data", "")
        try:
            parsed_json = json_repair.loads(raw_output)
        except:
            parsed_json = {"raw_text": raw_output}

        return {"status": "success", "data": parsed_json}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


if __name__ == "__main__":
    uvicorn.run("main:app", host="127.0.0.1", port=5000, reload=True)
